<a href="https://colab.research.google.com/github/rasushi/Fundus_AVSeg_Segmentation/blob/main/00_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

sushmithaveigas_fundus_avseg_path = kagglehub.dataset_download('sushmithaveigas/fundus-avseg')
sushmithaveigas_driave_path = kagglehub.dataset_download('sushmithaveigas/driave')

print('Data source import complete.')


In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter
import shutil
from tqdm import tqdm
import json

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"OpenCV version: {cv2.__version__}")

In [ ]:

DRIAVE_PATH = "/kaggle/input/driave/DRIAVE_DATASET"
FUNDUS_AVSEG_PATH = "/kaggle/input/fundus-avseg/Fundus-AVSeg"

# Output directory for processed data
OUTPUT_DIR = "/kaggle/working/processed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify paths exist
print("Checking dataset paths...")
print(f"DRIAVE exists: {os.path.exists(DRIAVE_PATH)}")
print(f"Fundus-AVSeg exists: {os.path.exists(FUNDUS_AVSEG_PATH)}")

if os.path.exists(DRIAVE_PATH):
    print("\nDRIAVE structure:")
    for item in os.listdir(DRIAVE_PATH)[:5]:
        print(f"  - {item}")

if os.path.exists(FUNDUS_AVSEG_PATH):
    print("\nFundus-AVSeg structure:")
    for item in os.listdir(FUNDUS_AVSEG_PATH)[:5]:
        print(f"  - {item}")

In [ ]:
def analyze_driave_mask(mask_path):
    """Analyze DRIAVE color-coded mask format"""
    mask = cv2.imread(mask_path)
    mask_rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

    # Count unique colors
    unique_colors = np.unique(mask_rgb.reshape(-1, 3), axis=0)

    print(f"\nAnalyzing: {os.path.basename(mask_path)}")
    print(f"Shape: {mask_rgb.shape}")
    print(f"Unique colors found: {len(unique_colors)}")

    # Identify which colors are present
    colors_dict = {}
    for color in unique_colors:
        # Check for red (artery)
        if color[0] > 200 and color[1] < 50 and color[2] < 50:
            colors_dict['red_artery'] = tuple(color)
        # Check for blue (vein)
        elif color[2] > 200 and color[0] < 50 and color[1] < 50:
            colors_dict['blue_vein'] = tuple(color)
        # Check for green (crossing)
        elif color[1] > 200 and color[0] < 50 and color[2] < 50:
            colors_dict['green_crossing'] = tuple(color)
        # Check for white (unknown)
        elif color[0] > 200 and color[1] > 200 and color[2] > 200:
            colors_dict['white_unknown'] = tuple(color)
        # Black (background)
        elif color[0] < 50 and color[1] < 50 and color[2] < 50:
            colors_dict['black_background'] = tuple(color)

    print("Colors present:")
    for name, rgb in colors_dict.items():
        pixel_count = np.sum(np.all(mask_rgb == rgb, axis=2))
        print(f"  {name}: {rgb} - {pixel_count} pixels")

    return mask_rgb, colors_dict

# Test on one DRIAVE mask
if os.path.exists(DRIAVE_PATH):
    sample_av_path = os.path.join(DRIAVE_PATH, "training", "av")
    if os.path.exists(sample_av_path):
        sample_masks = [f for f in os.listdir(sample_av_path) if f.endswith('.png')]
        if sample_masks:
            test_mask, colors_found = analyze_driave_mask(
                os.path.join(sample_av_path, sample_masks[0])
            )

In [ ]:
def convert_driave_mask_to_classes(color_mask):
    """
    Convert DRIAVE color-coded mask to 3-class segmentation

    Classes:
    0 = Background (black + white unknown + green crossing)
    1 = Artery (red)
    2 = Vein (blue)

    Args:
        color_mask: RGB image (H, W, 3)

    Returns:
        class_mask: Single channel (H, W) with values 0, 1, 2
    """
    h, w, _ = color_mask.shape
    class_mask = np.zeros((h, w), dtype=np.uint8)

    # Define color thresholds (RGB format)
    # Red (artery) - high R, low G, low B
    red_mask = (color_mask[:, :, 0] > 200) & \
               (color_mask[:, :, 1] < 50) & \
               (color_mask[:, :, 2] < 50)

    # Blue (vein) - low R, low G, high B
    blue_mask = (color_mask[:, :, 2] > 200) & \
                (color_mask[:, :, 0] < 50) & \
                (color_mask[:, :, 1] < 50)

    # Assign classes
    class_mask[red_mask] = 1  # Artery
    class_mask[blue_mask] = 2  # Vein
    # Everything else remains 0 (background)

    return class_mask

# Test the conversion
if os.path.exists(DRIAVE_PATH):
    sample_av_path = os.path.join(DRIAVE_PATH, "training", "av")
    if os.path.exists(sample_av_path):
        sample_masks = [f for f in os.listdir(sample_av_path) if f.endswith('.png')]
        if sample_masks:
            color_mask = cv2.imread(os.path.join(sample_av_path, sample_masks[0]))
            color_mask = cv2.cvtColor(color_mask, cv2.COLOR_BGR2RGB)

            class_mask = convert_driave_mask_to_classes(color_mask)

            print(f"\nConversion test:")
            print(f"Input shape: {color_mask.shape}")
            print(f"Output shape: {class_mask.shape}")
            print(f"Unique classes: {np.unique(class_mask)}")
            print(f"Class distribution:")
            for cls in [0, 1, 2]:
                count = np.sum(class_mask == cls)
                pct = (count / class_mask.size) * 100
                print(f"  Class {cls}: {count} pixels ({pct:.2f}%)")

In [ ]:
def visualize_mask_conversion(original_path, converted_mask):
    """Visualize original color mask and converted class mask"""
    original = cv2.imread(original_path)
    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Original color mask
    axes[0].imshow(original)
    axes[0].set_title("Original Color Mask\n(Red=Artery, Blue=Vein)")
    axes[0].axis('off')

    # Converted class mask
    axes[1].imshow(converted_mask, cmap='viridis')
    axes[1].set_title("Converted Class Mask\n(0=BG, 1=Artery, 2=Vein)")
    axes[1].axis('off')

    # Class mask with custom colors
    colored = np.zeros((*converted_mask.shape, 3), dtype=np.uint8)
    colored[converted_mask == 1] = [255, 0, 0]    # Red for artery
    colored[converted_mask == 2] = [0, 0, 255]    # Blue for vein
    axes[2].imshow(colored)
    axes[2].set_title("Colored Visualization")
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "mask_conversion_example.png"), dpi=150, bbox_inches='tight')
    plt.show()

# Visualize
if os.path.exists(DRIAVE_PATH):
    sample_av_path = os.path.join(DRIAVE_PATH, "training", "av")
    if os.path.exists(sample_av_path):
        sample_masks = [f for f in os.listdir(sample_av_path) if f.endswith('.png')]
        if sample_masks:
            mask_path = os.path.join(sample_av_path, sample_masks[0])
            color_mask = cv2.imread(mask_path)
            color_mask = cv2.cvtColor(color_mask, cv2.COLOR_BGR2RGB)
            class_mask = convert_driave_mask_to_classes(color_mask)
            visualize_mask_conversion(mask_path, class_mask)

In [ ]:
def process_driave_dataset(driave_path, output_dir):
    """
    Process DRIAVE dataset: convert images and create file lists

    Returns:
        train_files: list of (image_path, mask_path) for training
        test_files: list of (image_path, mask_path) for testing
    """
    train_files = []
    test_files = []

    print("Processing DRIAVE Dataset...")

    # Process training set
    train_img_dir = os.path.join(driave_path, "training", "images")
    train_av_dir = os.path.join(driave_path, "training", "av")

    if os.path.exists(train_img_dir) and os.path.exists(train_av_dir):
        train_images = sorted([f for f in os.listdir(train_img_dir) if f.endswith('.tif')])
        print(f"Found {len(train_images)} training images")

        for img_name in tqdm(train_images, desc="Processing DRIAVE training"):
            # Get corresponding mask (same name but .png)
            mask_name = img_name.replace('.tif', '.png')

            img_path = os.path.join(train_img_dir, img_name)
            mask_path = os.path.join(train_av_dir, mask_name)

            if os.path.exists(mask_path):
                train_files.append({
                    'image': img_path,
                    'mask': mask_path,
                    'dataset': 'DRIAVE',
                    'split': 'train',
                    'id': img_name.replace('.tif', '')
                })

    # Process test set
    test_img_dir = os.path.join(driave_path, "test", "images")
    test_av_dir = os.path.join(driave_path, "test", "av")

    if os.path.exists(test_img_dir) and os.path.exists(test_av_dir):
        test_images = sorted([f for f in os.listdir(test_img_dir) if f.endswith('.tif')])
        print(f"Found {len(test_images)} test images")

        for img_name in tqdm(test_images, desc="Processing DRIAVE test"):
            mask_name = img_name.replace('.tif', '.png')

            img_path = os.path.join(test_img_dir, img_name)
            mask_path = os.path.join(test_av_dir, mask_name)

            if os.path.exists(mask_path):
                test_files.append({
                    'image': img_path,
                    'mask': mask_path,
                    'dataset': 'DRIAVE',
                    'split': 'test',
                    'id': img_name.replace('.tif', '')
                })

    print(f"\nDRIAVE Summary:")
    print(f"  Training samples: {len(train_files)}")
    print(f"  Test samples: {len(test_files)}")

    return train_files, test_files

# Process DRIAVE
driave_train, driave_test = process_driave_dataset(DRIAVE_PATH, OUTPUT_DIR)

In [ ]:
def process_fundus_avseg_dataset(fundus_path, output_dir):
    """
    Process Fundus-AVSeg dataset using their train/test split

    Returns:
        train_files: list of dicts for training
        test_files: list of dicts for testing
    """
    train_files = []
    test_files = []

    print("\nProcessing Fundus-AVSeg Dataset...")

    images_dir = os.path.join(fundus_path, "images")
    masks_dir = os.path.join(fundus_path, "annotation")
    train_txt = os.path.join(fundus_path, "training.txt")
    test_txt = os.path.join(fundus_path, "testing.txt")

    # Read training split
    if os.path.exists(train_txt):
        with open(train_txt, 'r') as f:
            train_names = [line.strip() for line in f.readlines() if line.strip()]

        print(f"Found {len(train_names)} training images in training.txt")

        for img_name in tqdm(train_names, desc="Processing Fundus-AVSeg training"):
            img_path = os.path.join(images_dir, img_name)
            mask_path = os.path.join(masks_dir, img_name)

            if os.path.exists(img_path) and os.path.exists(mask_path):
                train_files.append({
                    'image': img_path,
                    'mask': mask_path,
                    'dataset': 'Fundus-AVSeg',
                    'split': 'train',
                    'id': img_name.replace('.png', '')
                })

    # Read test split
    if os.path.exists(test_txt):
        with open(test_txt, 'r') as f:
            test_names = [line.strip() for line in f.readlines() if line.strip()]

        print(f"Found {len(test_names)} test images in testing.txt")

        for img_name in tqdm(test_names, desc="Processing Fundus-AVSeg test"):
            img_path = os.path.join(images_dir, img_name)
            mask_path = os.path.join(masks_dir, img_name)

            if os.path.exists(img_path) and os.path.exists(mask_path):
                test_files.append({
                    'image': img_path,
                    'mask': mask_path,
                    'dataset': 'Fundus-AVSeg',
                    'split': 'test',
                    'id': img_name.replace('.png', '')
                })

    print(f"\nFundus-AVSeg Summary:")
    print(f"  Training samples: {len(train_files)}")
    print(f"  Test samples: {len(test_files)}")

    return train_files, test_files

# Process Fundus-AVSeg
fundus_train, fundus_test = process_fundus_avseg_dataset(FUNDUS_AVSEG_PATH, OUTPUT_DIR)

In [ ]:
def create_validation_split(train_files, val_ratio=0.15):
    """
    Create validation split from training data
    Stratified by dataset to maintain balance
    """
    # Separate by dataset
    driave_train = [f for f in train_files if f['dataset'] == 'DRIAVE']
    fundus_train = [f for f in train_files if f['dataset'] == 'Fundus-AVSeg']

    # Shuffle
    np.random.shuffle(driave_train)
    np.random.shuffle(fundus_train)

    # Split each dataset
    driave_val_size = int(len(driave_train) * val_ratio)
    fundus_val_size = int(len(fundus_train) * val_ratio)

    val_files = driave_train[:driave_val_size] + fundus_train[:fundus_val_size]
    train_files_final = driave_train[driave_val_size:] + fundus_train[fundus_val_size:]

    # Shuffle final splits
    np.random.shuffle(val_files)
    np.random.shuffle(train_files_final)

    return train_files_final, val_files

# Combine datasets
all_train = driave_train + fundus_train
all_test = driave_test + fundus_test

# Create validation split
train_final, val_final = create_validation_split(all_train, val_ratio=0.15)

print(f"\n{'='*60}")
print("FINAL DATASET SPLIT")
print(f"{'='*60}")
print(f"Training:   {len(train_final)} images")
print(f"Validation: {len(val_final)} images")
print(f"Test:       {len(all_test)} images")
print(f"{'='*60}")
print(f"Total:      {len(train_final) + len(val_final) + len(all_test)} images")

In [ ]:
def analyze_dataset_stats(train_files, val_files, test_files):
    """Analyze and visualize dataset statistics"""

    def get_stats(file_list, split_name):
        driave_count = sum(1 for f in file_list if f['dataset'] == 'DRIAVE')
        fundus_count = sum(1 for f in file_list if f['dataset'] == 'Fundus-AVSeg')

        print(f"\n{split_name} Split:")
        print(f"  DRIAVE: {driave_count} images ({driave_count/len(file_list)*100:.1f}%)")
        print(f"  Fundus-AVSeg: {fundus_count} images ({fundus_count/len(file_list)*100:.1f}%)")

        return driave_count, fundus_count

    print("\n" + "="*60)
    print("DATASET COMPOSITION")
    print("="*60)

    train_d, train_f = get_stats(train_files, "Training")
    val_d, val_f = get_stats(val_files, "Validation")
    test_d, test_f = get_stats(test_files, "Test")

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    splits = ['Training', 'Validation', 'Test']
    driave_counts = [train_d, val_d, test_d]
    fundus_counts = [train_f, val_f, test_f]

    for idx, (split, d_count, f_count) in enumerate(zip(splits, driave_counts, fundus_counts)):
        axes[idx].bar(['DRIAVE', 'Fundus-AVSeg'], [d_count, f_count],
                     color=['#3498db', '#e74c3c'])
        axes[idx].set_title(f'{split} Set\n(Total: {d_count + f_count})')
        axes[idx].set_ylabel('Number of Images')
        axes[idx].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "dataset_distribution.png"), dpi=150, bbox_inches='tight')
    plt.show()

analyze_dataset_stats(train_final, val_final, all_test)

In [ ]:
import json

def save_data_splits(train_files, val_files, test_files, output_dir):
    """Save data splits to JSON files"""

    # Save as JSON
    splits = {
        'train': train_files,
        'val': val_files,
        'test': test_files
    }

    for split_name, split_data in splits.items():
        output_path = os.path.join(output_dir, f'{split_name}_files.json')
        with open(output_path, 'w') as f:
            json.dump(split_data, f, indent=2)
        print(f"Saved {split_name} split: {output_path}")

    # Also save simplified text files for quick reference
    for split_name, split_data in splits.items():
        txt_path = os.path.join(output_dir, f'{split_name}_files.txt')
        with open(txt_path, 'w') as f:
            for item in split_data:
                f.write(f"{item['id']}\t{item['dataset']}\n")
        print(f"Saved {split_name} list: {txt_path}")

    # Save summary
    summary = {
        'total_images': len(train_files) + len(val_files) + len(test_files),
        'train_count': len(train_files),
        'val_count': len(val_files),
        'test_count': len(test_files),
        'datasets': ['DRIAVE', 'Fundus-AVSeg'],
        'num_classes': 3,
        'class_names': ['background', 'artery', 'vein']
    }

    summary_path = os.path.join(output_dir, 'dataset_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"Saved summary: {summary_path}")

# Save all splits
save_data_splits(train_final, val_final, all_test, OUTPUT_DIR)

In [ ]:
def create_sample_visualizations(train_files, val_files, test_files, output_dir, num_samples=3):
    """Create sample visualizations from each split"""

    def visualize_sample(file_dict, ax_row):
        # Load image
        if file_dict['image'].endswith('.tiff'):
            img = Image.open(file_dict['image'])
            img = np.array(img)
        else:
            img = cv2.imread(file_dict['image'])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Load mask
        mask = cv2.imread(file_dict['mask'])
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

        # Convert to class mask if from DRIAVE
        if file_dict['dataset'] == 'DRIAVE':
            class_mask = convert_driave_mask_to_classes(mask)
        else:
            # Fundus-AVSeg should already be in correct format, but verify
            class_mask = convert_driave_mask_to_classes(mask)

        # Display
        ax_row[0].imshow(img)
        ax_row[0].set_title(f"{file_dict['dataset']}\n{file_dict['id']}")
        ax_row[0].axis('off')

        ax_row[1].imshow(class_mask, cmap='viridis', vmin=0, vmax=2)
        ax_row[1].set_title("Class Mask")
        ax_row[1].axis('off')

        # Color overlay
        colored = np.zeros((*class_mask.shape, 3), dtype=np.uint8)
        colored[class_mask == 1] = [255, 0, 0]    # Red for artery
        colored[class_mask == 2] = [0, 0, 255]    # Blue for vein

        ax_row[2].imshow(img)
        ax_row[2].imshow(colored, alpha=0.5)
        ax_row[2].set_title("Overlay")
        ax_row[2].axis('off')

    # Sample from each split
    splits = {
        'Training': train_files,
        'Validation': val_files,
        'Test': test_files
    }

    for split_name, split_data in splits.items():
        # Sample images
        samples = np.random.choice(len(split_data), min(num_samples, len(split_data)), replace=False)

        fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
        if num_samples == 1:
            axes = axes.reshape(1, -1)

        fig.suptitle(f'{split_name} Set Samples', fontsize=16, fontweight='bold')

        for idx, sample_idx in enumerate(samples):
            visualize_sample(split_data[sample_idx], axes[idx])

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'samples_{split_name.lower()}.png'),
                   dpi=150, bbox_inches='tight')
        plt.show()

# Create visualizations
create_sample_visualizations(train_final, val_final, all_test, OUTPUT_DIR, num_samples=3)

In [ ]:
print("\n" + "="*80)
print("DATA PREPARATION COMPLETE!")
print("="*80)

print(f"\nOutput Directory: {OUTPUT_DIR}")
print(f"\nFiles created:")
print(f"  ✅ train_files.json ({len(train_final)} samples)")
print(f"  ✅ val_files.json ({len(val_final)} samples)")
print(f"  ✅ test_files.json ({len(all_test)} samples)")
print(f"  ✅ dataset_summary.json")
print(f"  ✅ Sample visualizations")

print(f"\n📊 Dataset Statistics:")
print(f"  Training:   {len(train_final):>4} images ({len(train_final)/(len(train_final)+len(val_final)+len(all_test))*100:.1f}%)")
print(f"  Validation: {len(val_final):>4} images ({len(val_final)/(len(train_final)+len(val_final)+len(all_test))*100:.1f}%)")
print(f"  Test:       {len(all_test):>4} images ({len(all_test)/(len(train_final)+len(val_final)+len(all_test))*100:.1f}%)")
print(f"  {'─'*40}")
print(f"  Total:      {len(train_final)+len(val_final)+len(all_test):>4} images")

print(f"\n🎯 Next Steps:")
print(f"  1. Review sample visualizations to verify data quality")
print(f"  2. Check that masks are converted correctly")
print(f"  3. Proceed to Model Training ")

# Create a config file for training
config = {
    'data': {
        'train_json': os.path.join(OUTPUT_DIR, 'train_files.json'),
        'val_json': os.path.join(OUTPUT_DIR, 'val_files.json'),
        'test_json': os.path.join(OUTPUT_DIR, 'test_files.json'),
        'num_classes': 3,
        'class_names': ['background', 'artery', 'vein']
    },
    'training': {
        'image_size': 512,
        'batch_size': 8,
        'num_epochs': 30,
        'learning_rate': 0.0001,
        'early_stopping_patience': 5
    }
}

config_path = os.path.join(OUTPUT_DIR, 'training_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"\n✅ Training config saved: {config_path}")
print("\n" + "="*80)